# UC02 — Predicción de Turbidez y Parámetros Fisicoquímicos**Objetivo:** Predecir turbidez, pH, cloro residual y conductividad a 4-24 horas vista para ajustar proactivamente la dosificación.**Tecnologías:** SNOWFLAKE.ML.FORECAST · Streamlit**Tiempo estimado:** 14 minutos

## Paso 1 — Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS PREDICCION_CALIDAD_DB;USE DATABASE PREDICCION_CALIDAD_DB;CREATE SCHEMA IF NOT EXISTS PUBLIC;USE SCHEMA PUBLIC;CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;USE WAREHOUSE COMPUTE_WH;

## Paso 2 — Generar Series Temporales de CalidadGeneramos 365 días de lecturas horarias (8.760 registros) con patrones estacionales y correlaciones realistas.

In [ ]:
CREATE OR REPLACE TABLE lecturas_calidad ASSELECT    DATEADD('hour', -SEQ4(), CURRENT_TIMESTAMP()) AS timestamp,    ROUND(GREATEST(1, 8 + 15 * SIN(SEQ4() * 3.14159 / 4380) + UNIFORM(-5, 20, RANDOM()) + CASE WHEN UNIFORM(0,100,RANDOM()) < 5 THEN UNIFORM(30, 100, RANDOM()) ELSE 0 END), 2) AS turbidez_ntu,    ROUND(7.2 + 0.3 * SIN(SEQ4() * 3.14159 / 2190) + UNIFORM(-3, 3, RANDOM()) * 0.1, 2) AS ph,    ROUND(GREATEST(0.1, 0.8 - 0.2 * SIN(SEQ4() * 3.14159 / 4380) + UNIFORM(-2, 2, RANDOM()) * 0.1), 2) AS cloro_residual_mg_l,    ROUND(400 + 100 * SIN(SEQ4() * 3.14159 / 4380) + UNIFORM(-30, 30, RANDOM()), 1) AS conductividad_us_cm,    ROUND(10 + 8 * SIN(SEQ4() * 3.14159 / 4380) + UNIFORM(-2, 2, RANDOM()), 1) AS temperatura_agua,    ROUND(200 + 80 * SIN(SEQ4() * 3.14159 / 8760) + UNIFORM(-30, 30, RANDOM()), 1) AS caudal_bruto_m3hFROM TABLE(GENERATOR(ROWCOUNT => 8760));SELECT COUNT(*) AS registros, MIN(timestamp) AS desde, MAX(timestamp) AS hasta FROM lecturas_calidad;

## Paso 3 — Entrenar Modelo de Predicción de TurbidezUsamos SNOWFLAKE.ML.FORECAST para predecir la turbidez a 24 horas vista.

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST predictor_turbidez(    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'lecturas_calidad'),    TIMESTAMP_COLNAME => 'timestamp',    TARGET_COLNAME => 'turbidez_ntu');

## Paso 4 — Generar PrediccionesObtenemos las predicciones para las próximas 24 horas con intervalos de confianza.

In [ ]:
CALL predictor_turbidez!FORECAST(FORECASTING_PERIODS => 24);CREATE OR REPLACE TABLE predicciones_turbidez ASSELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));SELECT * FROM predicciones_turbidez ORDER BY TS LIMIT 24;

## Paso 5 — Alertas Regulatorias ProactivasIdentificamos predicciones que superan los umbrales del RD 140/2003 (turbidez máxima = 4 NTU).

In [ ]:
CREATE OR REPLACE TABLE alertas_proactivas ASSELECT    TS AS timestamp_predicho,    FORECAST AS turbidez_predicha,    LOWER_BOUND AS intervalo_inferior,    UPPER_BOUND AS intervalo_superior,    CASE WHEN FORECAST > 4 THEN 'ALERTA: Supera umbral regulatorio (4 NTU)'         WHEN FORECAST > 2 THEN 'AVISO: Turbidez elevada, monitorizar'         ELSE 'Normal' END AS estado_regulatorioFROM predicciones_turbidezWHERE FORECAST > 2;SELECT * FROM alertas_proactivas ORDER BY timestamp_predicho;

## Paso 6 — Dashboard de PrediccionesPanel interactivo con gráficos de tendencia, predicciones y alertas.

In [ ]:
import streamlit as stfrom snowflake.snowpark.context import get_active_sessionimport pandas as pdsession = get_active_session()st.title("📈 Predicción de Turbidez — Panel de Calidad del Agua")col1, col2, col3 = st.columns(3)pred = session.sql("SELECT * FROM predicciones_turbidez ORDER BY TS").to_pandas()alertas = session.sql("SELECT COUNT(*) AS n FROM alertas_proactivas").collect()[0]["N"]ultima = session.sql("SELECT turbidez_ntu FROM lecturas_calidad ORDER BY timestamp DESC LIMIT 1").collect()[0]["TURBIDEZ_NTU"]col1.metric("Turbidez Actual", f"{ultima} NTU")col2.metric("Predicción 24h", f"{pred['FORECAST'].iloc[-1]:.1f} NTU")col3.metric("Alertas Activas", alertas)st.subheader("Predicción a 24 horas")st.line_chart(pred.set_index("TS")[["FORECAST", "LOWER_BOUND", "UPPER_BOUND"]])st.subheader("Alertas Regulatorias")st.dataframe(session.sql("SELECT * FROM alertas_proactivas ORDER BY timestamp_predicho").to_pandas())

## Paso 7 — Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS PREDICCION_CALIDAD_DB;-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;